## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.

LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

```python
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# Define schema
class Person(BaseModel):
    name: str = Field(description="Person's name")
    age: int = Field(description="Person's age")
    city: str = Field(description="City where the person lives")

# Initialize model
model = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

# Enable structured output
structured_model = model.with_structured_output(Person)

# Invoke model
response = structured_model.invoke(
    "John is 25 years old and lives in Chennai."
)

print(response)
```

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

c:\Users\DELL\Documents\AgenticAI\langchain_updated\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from pydantic import BaseModel, Field  # it enbles the scheme for the AI so, it can understand the description provide it answer

class Movie(BaseModel):
    title: str = Field(
        description="The title of the movie"
    )

    year: int = Field(
        description="The year the movie was released"
    )

    director: str = Field(
        description="The director of the movie"
    )

    rating: float = Field(
        description="The movie rating out of 10"
    )


In [3]:
# Enable structured output
model_with_structure = model.with_structured_output(Movie)

# Generate structured response
response = model_with_structure.invoke(
    "provide detail abput the movie avatar"
)

print(response)

title='Avatar' year=2009 director='James Cameron' rating=7.8


In [4]:
### Normal invoke model it is just for comparations with structure scheme with normal output

model.invoke("provide detail abput the movie avatar")

AIMessage(content='<think>\nOkay, I need to explain the movie "Avatar" in detail. Let me start by recalling what I know. It\'s directed by James Cameron, right? It came out in 2009, I think. The main plot involves a human named Jake Sully who goes to a planet called Pandora. He uses a body called an avatar to interact with the environment and the indigenous people, the Na\'vi. There\'s a conflict between the humans and the Na\'vi over a mineral called Unobtanium. \n\nWait, the humans are there to mine that mineral, and the Na\'vi are defending their home. Jake becomes connected to the Na\'vi and helps them fight against the humans. The movie is known for its groundbreaking visual effects and 3D technology. It\'s also a science fiction film with themes of environmentalism and indigenous rights. \n\nI should mention the main characters. Jake Sully is the protagonist, and then there\'s his twin brother, who was supposed to go to Pandora but died in an accident. That\'s why Jake is there. 

### Message output alongside parse Structure 

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(
        ...,
        description="The title of the movie"
    )

    year: int = Field(
        ...,
        description="The year the movie was released"
    )

    director: str = Field(
        ...,
        description="The director of the movie"
    )

    rating: float = Field(
        ...,
        description="The movie's rating out of 10"
    )

# Enable structured output with raw response
model_with_structure = model.with_structured_output(
    Movie,
    include_raw=True
)

# Invoke model
response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I need to provide those parameters. \n\nFirst, I know that Inception is directed by Christopher Nolan. The release year was 2010. The title is obviously "Inception". For the rating, I should check a reliable source like IMDb. Inception has an 8.8/10 rating on IMDb, so I\'ll use that. \n\nWait, do I need to confirm the exact rating? Maybe, but I think 8.8 is correct. Let me double-check. Yes, IMDb does list it as 8.8. Alright, that\'s the data. Now, I need to structure the function call with these details. Make sure all required fields are included: title, year, director, rating. No missing parameters. \n\nLet me format the JSON properly. The name of the function is "Movie", and the arguments should be a JSON object with the keys and 

Nested Structure

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]

    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )


# Enable structured output
model_with_structure = model.with_structured_output(
    MovieDetails
)

# Invoke model
response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')] genres=['Science Fiction', 'Action', 'Thriller'] budget=160.0


## TypedDict

`TypedDict` provides a simpler alternative using Python's built-in typing system. It is ideal when you don't need runtime validation or advanced Pydantic features.

```python
from typing import TypedDict

class Movie(TypedDict):
    title: str
    year: int
    director: str
    rating: float

# Enable structured output
model_with_structure = model.with_structured_output(Movie)

# Invoke model
response = model_with_structure.invoke(
    "Provide details about the movie Interstellar"
)

print(response)
```

## TypedDict vs BaseModel

`TypedDict` and `BaseModel` are both used to define the structure (schema) of data for structured outputs in LangChain, but they serve different purposes.

## Why We Use Structured Output

Normally, LLM responses are plain text:

```python
"Interstellar was released in 2014..."
```

This is hard to:
- Parse reliably
- Store in databases
- Use in APIs
- Validate automatically

Structured output forces the model to return data in a predictable format:

```python
{
    "title": "Interstellar",
    "year": 2014,
    "rating": 8.7
}
```

This makes AI outputs:
- Predictable
- Machine-readable
- Easier for automation
- Safer for production apps

---

# TypedDict

`TypedDict` is a lightweight schema system from Python typing.

```python
from typing import TypedDict

class Movie(TypedDict):
    title: str
    year: int
```

It mainly provides:
- Type hints
- Structure definition
- Editor autocomplete
- Static type checking

However, it does NOT provide:
- Runtime validation
- Field descriptions
- Automatic constraints

## Why Use TypedDict?

Use it when you want:
- Simple schemas
- Less overhead
- Faster lightweight code
- Basic structured output

Good for:
- Small projects
- Prototypes
- Simple AI extraction

---

# BaseModel (Pydantic)

`BaseModel` from Pydantic is much more powerful.

```python
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str
    year: int
```

It provides:
- Runtime validation
- Automatic type conversion
- Field descriptions
- Nested models
- Constraints
- Serialization
- Error handling

---

# Key Difference

## TypedDict

If the model returns:

```python
{
    "title": "Inception",
    "year": "2010"
}
```

TypedDict:
- Accepts it silently
- Does not validate types

---

## BaseModel

Pydantic automatically converts:

```python
"2010" -> 2010
```

and validates types.

If invalid:

```python
{
    "year": "abc"
}
```

it raises validation errors.

---

# Feature Comparison

| Feature | TypedDict | BaseModel |
|---|---|---|
| Type hints | ✅ | ✅ |
| Runtime validation | ❌ | ✅ |
| Field descriptions | ❌ | ✅ |
| Nested validation | Basic | Advanced |
| Serialization | ❌ | ✅ |
| Error handling | ❌ | ✅ |
| Lightweight | ✅ | ❌ |
| Production ready | Limited | Excellent |

---

# Real AI Agent Usage

## TypedDict
Good for:
- Quick extraction
- Lightweight agents
- Simple outputs

## BaseModel
Best for:
- Production AI systems
- APIs
- Multi-agent workflows
- Structured pipelines
- RAG systems
- Databases

---

# Example Use Case

Suppose your stock market NLP project extracts:

```python
{
    "ticker": "AAPL",
    "sentiment": "positive",
    "confidence": 0.92
}
```

Using `BaseModel` ensures:
- `confidence` is a float
- `ticker` exists
- `sentiment` format is valid

This becomes very important in real AI systems.

---

# Simple Analogy

## TypedDict
Like:
```text
"Expected structure"
```

## BaseModel
Like:
```text
"Structure + validator + security guard"
```

---

# Recommendation

If you are learning:
- AI agents
- LangChain
- Structured workflows
- NLP pipelines

focus mainly on:
- `BaseModel`
- Pydantic
- Structured outputs

because modern AI systems rely heavily on them.

### DataClasses

In [7]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [12]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_groq import ChatGroq


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str          # The name of the person
    email: str         # The email address of the person
    phone: str         # The phone number of the person


# Create model
model = ChatGroq(
    model="qwen/qwen3-32b"
)


# Create agent
agent = create_agent(
    model=model,
    response_format=ContactInfo
)


# Invoke agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Extract contact info from: "
                    "John Doe, john@example.com, "
                    "(555) 123-4567"
                )
            }
        ]
    }
)


# Print structured output
print(result["structured_response"])

# Output:
# {
#     'name': 'John Doe',
#     'email': 'john@example.com',
#     'phone': '(555) 123-4567'
# }

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [ ]:
from dataclasses import dataclass
from langchain.agents import 